# 05 · Por Trás dos Panos

**Teoria**: docs/02-rdds-linhagem-particoes.md, docs/03-transformacoes-acoes-dag.md, 
docs/04-dataframes-catalyst-tungsten.md, docs/06-persistencia-e-otimizacao.md

🎯 **Objetivo deste notebook**: levantar o capô do Spark e explorar seus mecanismos internos — 
lineage de RDD, plano Catalyst, cache e comparação honesta com Pandas.

Você já confia nos resultados dos notebooks 01-04. Agora: **por que** o Spark 
levou aquele tempo pra calcular, e **como** ele decide o plano de execução?

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)

spark

## Lazy evaluation e lineage de RDD

💡 **Conceito fundamental**: nada abaixo executa até uma **ação** (`collect`, `sum`, ...) ser chamada.
Transformações (`filter`, `map`) só ficam registradas no grafo de 
lineage — o Spark as executa de forma **preguiçosa** (lazy evaluation) para poder 
otimizar o pipeline como um todo antes de disparar a execução.

⚠️ **Atenção**: se você chamar apenas `filter().map()`, nada acontece no cluster. 
O Spark só sai da inércia quando uma ação é invocada. Isso permite que o Catalyst 
enxergue o pipeline completo e decida a melhor ordem de execução.

In [ ]:
# Obtém o SparkContext de baixo nível (necessário para criar e manipular RDDs)
sc = spark.sparkContext

# Cria um RDD particionado em 8 fatias, cada uma com ~125.000 números
# numSlices controla o paralelismo: cada partition vira uma Task executada por um core
numbers = sc.parallelize(range(1, 1_000_001), numSlices=8)
print(f"Partições: {numbers.getNumPartitions()}")

# Transformações — são preguiçosas, nada roda ainda
evens = numbers.filter(lambda n: n % 2 == 0)    # Filtra apenas números pares
squared = evens.map(lambda n: n * n)             # Eleva cada par ao quadrado

In [ ]:
# Ação — É AQUI que o Spark de fato constrói o DAG e executa
# O sum() força o Spark a percorrer todo o grafo de dependências
# Cada partition computa sua soma parcial, depois o driver reduz (soma) todos os parciais
total = squared.sum()
print(f"Soma dos quadrados dos pares de 1..1.000.000: {total:,}")

📌 **Observação sobre o resultado**:

O Spark executou o pipeline completo: `range → filter → map → sum`. Cada partition foi 
processada independentemente por uma Task, e o `sum()` agregou os resultados parciais (fase reduce).

💡 **Dica**: experimente variar `numSlices` (ex.: 2, 8, 64) e observe o efeito no tempo. 
Com poucos dados, muitas partições podem *piorar* a performance (overhead de agendamento).

⚠️ **Atenção**: `squared` ainda é um RDD — não podemos usar `.explain()` como faríamos com DataFrames.

In [ ]:
# O grafo de lineage que o Spark usaria para recomputar este RDD em caso de falha:
# toDebugString() mostra a genealogia como uma árvore indentada
# Cada linha com parênteses representa um RDD ancestral e a transformação aplicada
# Quanto mais profundo o grafo, mais caro seria recomputar após uma falha de nó
print(squared.toDebugString().decode())

🧠 **Entendendo o grafo de lineage**:

A árvore acima mostra a genealogia do RDD `squared`. Cada indentação representa uma dependência. 
Se uma partition for perdida (falha de nó worker), o Spark pode recomputá-la percorrendo essa 
linhagem — sem precisar de replicação.

📌 **Limitação**: o lineage de RDD é **opaco** — mostra apenas a sequência de transformações, 
sem otimização. É por isso que o DataFrame (próxima seção) é superior: ele expõe o schema ao 
Catalyst, que pode reordenar, podar e otimizar o plano de execução.

#### 💡 **Exemplo 1:** Espiando dentro de um RDD — como os dados ficam distribuídos entre partições

Até aqui falamos de "8 partições" como um número abstrato. `glom()` torna isso concreto: agrupa os elementos de **cada partição** numa lista Python, deixando ver exatamente quais dados moram em qual partição — algo que normalmente fica escondido dentro dos executores.

In [ ]:
# RDD pequeno e didático — 20 números em 4 partições, fácil de visualizar por completo
amostra = sc.parallelize(range(1, 21), numSlices=4)

# glom(): agrupa os elementos de CADA partição em uma lista Python.
# Normalmente "olhar lá dentro" de uma partição exigiria acessar cada executor —
# glom() + collect() traz essa visão inteira pro Driver de uma vez
particoes = amostra.glom().collect()
for i, particao in enumerate(particoes):
    print(f"Partição {i}: {particao}")

📌 **O que isso revela:** `range(1, 21)` com `numSlices=4` foi dividido em 4 blocos **contíguos** de 5 elementos cada — é assim que o Spark particiona uma coleção Python: fatias sequenciais, uma por partição. Cada partição vira uma **Task** que roda, em paralelo, num core diferente.

⚠️ **Cuidado:** `glom()` traz TODOS os dados para o Driver, igual `collect()` — ótimo para aprender com uma amostra pequena, perigoso em produção com dados grandes (o mesmo aviso que já vimos para `collect()`/`toPandas()` nos notebooks anteriores).

#### 💡 **Exemplo 2:** Lineage sob a lupa — por que `evens` "sumiu", e o que aparece quando existe shuffle

O `toDebugString()` de `squared` (algumas células acima) tem só **2 linhas**, mesmo vindo de duas transformações (`filter` seguido de `map`). Vamos provar por que — e mostrar o que muda quando entra um shuffle de verdade.

In [ ]:
# Cada RDD Python tem um id() próprio — mesmo quando nunca "aparece" no plano físico
print(f"numbers.id() = {numbers.id()}")
print(f"evens.id()   = {evens.id()}")
print(f"squared.id() = {squared.id()}")
print()
print(squared.toDebugString().decode())

📌 **A prova:** `evens` tem seu próprio `id` — é um objeto RDD Python **distinto** de `squared`. Mas ele não aparece como uma linha própria no `toDebugString()`. Isso é **pipelining**: como `filter` e `map` são transformações **narrow** (nenhuma delas precisa reunir dados de outras partições), o Spark funde as duas numa única função `filter→map` que roda de uma vez, partição por partição — sem nunca materializar `evens` em lugar nenhum.

In [ ]:
# reduceByKey EXIGE shuffle: valores da mesma chave precisam se encontrar,
# possivelmente vindos de partições (e executores) diferentes
pares_por_resto = squared.map(lambda n: (n % 10, n))
soma_por_resto = pares_por_resto.reduceByKey(lambda a, b: a + b)

print(soma_por_resto.toDebugString().decode())

📌 **Agora o lineage "fala":** 5 linhas, não mais 2. Repare no `+-` — ele marca a **fronteira do shuffle**: tudo acima roda depois dos dados serem embaralhados por chave; tudo abaixo (`PairwiseRDD`, `PythonRDD`, `ParallelCollectionRDD`) roda no estágio anterior. `ShuffledRDD` é o Spark dizendo, sem rodeios, "aqui os dados trocaram de partição pela rede".

🧠 **Regra prática:** conte os `+-` (ou os saltos de indentação) no `toDebugString()` para estimar quantos estágios — e quantos shuffles — o seu job vai ter. Cada um é um ponto caro de sincronização entre todas as partições, exatamente como vimos com `Exchange` nos planos Catalyst do notebook 04.

## RDD vs. DataFrame: o plano que o Catalyst enxerga

🎯 **Objetivo**: comparar o lineage opaco de RDD com o **plano Catalyst** de um DataFrame.

Mesma lógica de `filter → map → sum`, mas o DataFrame dá ao Spark um **schema** — 
é isso que torna a otimização do Catalyst possível.

📌 O `explain(True)` (modo estendido) mostra: **plano lógico não resolvido** → 
**plano lógico resolvido** → **plano físico** com os operadores reais que 
serão executados nos executores.

In [ ]:
# Cria um DataFrame com 1 milhão de números — há schema e tipagem, diferente do RDD
# .toDF("n") nomeia a única coluna como "n"
df = spark.range(1, 1_000_001).toDF("n")

# Filtra pares e calcula soma dos quadrados usando expressão SQL
# O Catalyst pode: empurrar o filtro (filter pushdown), podar colunas não usadas, etc.
resultado = df.filter(df.n % 2 == 0).selectExpr("sum(n * n) as soma_quadrados")

# explain(True) = extended mode: plano lógico + físico + otimizações aplicadas
# Compare a riqueza de informação com o toDebugString() do RDD acima
resultado.explain(True)
resultado.show()

📌 **Comparando os planos**:

Enquanto o lineage do RDD mostrava `MapPartitionsRDD[2] ← MapPartitionsRDD[1] ← ...`, 
o plano Catalyst do DataFrame revela operadores nomeados:

- `Scan` — leitura com estatísticas de tamanho
- `Filter` — predicate pushdown (quando aplicável)
- `HashAggregate` — agregação parcial + final (combine antes do shuffle)
- `Exchange` — shuffle dos dados entre partições (ponto mais caro do plano)
- `SerializeFromObject / DeserializeToObject` — serialização binária via Tungsten

💡 **Dica**: quanto mais informação o Catalyst tem (schema, estatísticas, dicas de broadcast), 
melhor o plano físico gerado.

#### 💡 **Exemplo 3:** De DataFrame a RDD — o que tem por baixo do Catalyst

Todo DataFrame Spark tem um RDD por baixo dele, acessível via `.rdd`. Vamos abrir essa tampa e comparar com o RDD "cru" que já exploramos.

In [ ]:
print("Tipo de df.rdd:", type(df.rdd))
print("Partições do RDD por trás do DataFrame:", df.rdd.getNumPartitions())
print("Partições do nosso RDD manual (numbers):", numbers.getNumPartitions())
print()
print("Primeiro elemento:", df.rdd.first(), "-- tipo:", type(df.rdd.first()).__name__)

📌 **Já duas surpresas:**

1. **Partições diferentes por padrão:** não escolhemos o número de partições de `df` — o Spark usou `sc.defaultParallelism` (o número de núcleos da sua máquina), bem diferente do `numSlices=8` que escolhemos manualmente para `numbers`. RDD exige que você pense em particionamento explicitamente; DataFrame decide por você (mas ainda dá pra sobrescrever com `.repartition()`).
2. **Cada elemento virou um `Row`:** o formato binário otimizado do Tungsten (colunar, fora da heap da JVM) precisa ser **desserializado** de volta para objetos Python só para você conseguir tocar nele via `.rdd`.

In [ ]:
print(df.rdd.toDebugString().decode())

📌 **Compare com o `squared.toDebugString()` de duas seções atrás:** lá eram 2 linhas; aqui são 6 — várias `MapPartitionsRDD` e um `SQLExecutionRDD` que não existiam antes. Esse é o custo escondido de `.rdd`: o Catalyst precisa desmontar seu plano físico otimizado (binário, Tungsten) e reconstruir cada linha como um objeto Python `Row`, partição por partição.

In [ ]:
import time

# Catalyst: expressão nativa — os dados nunca saem do formato binário Tungsten
start = time.perf_counter()
soma_catalyst = df.filter(df.n % 2 == 0).selectExpr("sum(n * n) as soma").first()["soma"]
tempo_catalyst = time.perf_counter() - start

# .rdd: cada linha vira um objeto Row em Python — filter/map rodam em Python puro, linha a linha
start = time.perf_counter()
soma_via_rdd = df.rdd.filter(lambda row: row.n % 2 == 0).map(lambda row: row.n * row.n).sum()
tempo_rdd = time.perf_counter() - start

print(f"Catalyst (DataFrame API):     {tempo_catalyst:.3f}s -> {soma_catalyst:,}")
print(f"Via .rdd (Row a Row, Python): {tempo_rdd:.3f}s -> {soma_via_rdd:,}")

📌 **O mesmo resultado, caminhos bem diferentes:** processar via `.rdd` foi mais lento — cada partição precisou desserializar cada linha em um objeto `Row` Python antes de aplicar `filter`/`map`, o oposto do que o Tungsten faz (processar em formato binário, sem sair da JVM). Essa é a razão prática por trás de uma recomendação que já apareceu nos notebooks anteriores: prefira sempre a API de DataFrame — `.rdd` é uma porta de escape para quando você realmente precisa de controle fino, não o caminho padrão.

🧠 **Fechando o ciclo:** RDD é a fundação (partições, lineage, tasks); DataFrame é a camada que dá **schema** para o Catalyst otimizar em cima dela. Por baixo dos panos, é RDD o tempo todo — a diferença é o quanto de informação o Spark tem disponível para tomar decisões melhores.

## O Catalyst nos joins do notebook 04

📌 `BroadcastHashJoin` (sem Shuffle da tabela grande) vs. `SortMergeJoin` 
(Shuffle dos dois lados) — o mesmo par de junções do notebook 04, agora 
olhando o **plano de execução** por trás delas.

🧠 **Por quê?**: o tamanho das tabelas é o fator decisivo. `empresas` (~50 linhas) 
cabe na memória de cada executor e pode ser broadcastada. `funcionarios` 
(milhares de linhas) exige que ambos os lados sejam reparticionados por 
`id_funcionario` e shufflados entre os nós.

In [ ]:
from pyspark.sql.functions import broadcast

# Lê os dados das camadas Bronze — cada arquivo Parquet vira um DataFrame com schema inferido
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("../data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("../data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("../data/bronze/vendas")

# Broadcast join: dica explícita (broadcast()) para o Catalyst usar BroadcastHashJoin
# empresas (~50 linhas) é copiada para CADA executor — sem shuffle do lado grande vendas
join_broadcast = sdf_vendas.join(broadcast(sdf_empresas), "id_empresa")
print("--- broadcast join (empresas, 50 linhas) ---")
join_broadcast.explain()

# Shuffle join: SortMergeJoin — ambos os lados são shufflados por id_funcionario
# Funcionarios tem milhares de linhas, então não cabe no limite de broadcast (10 MB default)
join_shuffle = sdf_vendas.join(sdf_funcionarios, "id_funcionario")
print("--- shuffle join (funcionarios, milhares de linhas) ---")
join_shuffle.explain()

📌 **Interpretando os planos de join**:

No primeiro plano, procure por `BroadcastHashJoin`. A palavra "Broadcast" significa que 
a tabela `empresas` será copiada integralmente para TODOS os executores — o lado grande 
(`vendas`, centenas de milhares de linhas) não sofre shuffle.

No segundo plano, procure por `SortMergeJoin` e dois nós `Exchange`. 
Cada `Exchange` = um shuffle: um para `vendas` por `id_funcionario`, outro para 
`funcionarios` pelo mesmo campo. O custo de rede é significativamente maior.

⚠️ **Atenção**: Catalyst *pode* escolher BroadcastHashJoin automaticamente se a tabela pequena 
for menor que `spark.sql.autoBroadcastJoinThreshold` (10 MB por padrão).

## Spark SQL — mesmo motor, sintaxe SQL

💡 O Catalyst é o **mesmo otimizador**, independente da API usada (DataFrame, SQL ou RDD). 
Aqui usamos SQL puro — útil para equipes que já dominam SQL ou para migração gradual 
de data warehouses legados.

📌 Toda consulta SQL passa pelo mesmo pipeline: parser → análise lógica → Catalyst → 
plano físico → execução no cluster.

In [ ]:
# Registra DataFrames como tabelas temporárias — visíveis apenas nesta sessão Spark
# createOrReplaceTempView substitui a view se já existir com o mesmo nome
sdf_vendas.createOrReplaceTempView("vendas")
sdf_empresas.createOrReplaceTempView("empresas")
sdf_funcionarios.createOrReplaceTempView("funcionarios")

# Consulta SQL pura — o Catalyst resolve, otimiza e executa (mesmo plano que DataFrame API)
# Junta vendas → funcionarios → empresas, agrupa por setor, ordena por total e limita 10
resultado_sql = spark.sql("""
    SELECT e.setor, SUM(v.valor) AS total_vendas
    FROM vendas v
    JOIN funcionarios f ON v.id_funcionario = f.id_funcionario
    JOIN empresas e ON f.id_empresa = e.id_empresa
    GROUP BY e.setor
    ORDER BY total_vendas DESC
    LIMIT 10
""")
resultado_sql.show()

🧠 **O que aprender aqui**:

O resultado é idêntico ao que você obteria com DataFrame API. O Catalyst não diferencia 
a origem da consulta — ele sempre gera o mesmo plano físico otimizado.

💡 **Dica**: use `explain()` também em consultas SQL para depurar performance: 
`spark.sql("EXPLAIN SELECT ...").show(truncate=False)` mostra o plano completo sem executar.

## Cache: pagar uma vez, reusar muitas

🎯 **Objetivo**: demonstrar na prática como o cache evita reexecutar transformações caras.

O primeiro `count()` lê o Parquet do zero e aplica o filtro `valor > 50`. 
O segundo processamento (`groupBy ano.count`) roda sobre o dado já cacheado em 
memória — sem reler o disco.

⚠️ **Atenção**: cache só vale a pena se o mesmo DataFrame for reutilizado em múltiplas ações. 
Cachear um DataFrame usado uma única vez adiciona overhead desnecessário (serialização + memória).

In [ ]:
import time

from pyspark.sql.functions import col

# Aplica um filtro — transformação lazy, nada executou ainda
vendas_filtradas = sdf_vendas.filter(col("valor") > 50)

# Primeira ação: força leitura do Parquet + aplicação do filtro + contagem
# Ainda sem cache — toda ação futura reexecutará o pipeline do zero
start = time.perf_counter()
contagem_1 = vendas_filtradas.count()
sem_cache_segundos = time.perf_counter() - start

# Ativa cache (padrão: StorageLevel.MEMORY_ONLY) e materializa com um count()
# Após este count(), os dados filtrados residem em memória nos executores
vendas_filtradas.cache()
vendas_filtradas.count()  # materializa o cache

# Segunda consulta: groupBy ano — executa sobre o cache, sem reler o Parquet
start = time.perf_counter()
contagem_2 = vendas_filtradas.groupBy("ano").count().count()
com_cache_segundos = time.perf_counter() - start

print(f"Primeiro count() (sem cache ainda): {sem_cache_segundos:.3f}s")
print(f"groupBy sobre dado cacheado:         {com_cache_segundos:.3f}s")

# Libera a memória do cache — boa prática para não reter recursos desnecessariamente
vendas_filtradas.unpersist()

📌 **Análise dos tempos**:

Compare `sem_cache_segundos` vs `com_cache_segundos`.

O primeiro tempo inclui: leitura do Parquet no disco + descompressão + aplicação do filtro + contagem.
O segundo tempo inclui apenas: leitura da memória + groupBy + contagem.

💡 **Dica**: abra a aba **Storage** da Spark UI (http://localhost:4040/storage/) para ver 
o DataFrame cacheado, seu tamanho em memória e a fração de armazenamento utilizada.

## PySpark vs. Pandas nesta escala

🧠 **Por quê?** — em algumas centenas de milhares de linhas, o Pandas não paga overhead de 
distribuição/serialização — é bem provável que vença aqui.

A lição não é "Spark é lento", é **"use a ferramenta certa pro tamanho do dado"** (ver 
docs/05). O Spark brilha em datasets que não cabem na memória de uma única máquina.

📌 Essa mesma comparação muda de lado à medida que o volume cresce e o trabalho passa a 
ser distribuído entre múltiplos nós reais — na casa de centenas de milhões de linhas, o 
overhead de coordenação do Spark deixa de ser o fator dominante e a distribuição passa a compensar.

In [ ]:
# Converte o DataFrame Spark para Pandas — CUIDADO: traz todos os dados para o driver!
# Só é viável quando o dataset cabe na memória RAM do driver (não escala para grandes volumes)
pdf = sdf_vendas.toPandas()

# Pandas: groupBy + sum em memória local (single-node, dados em formato numpy nativo)
# Sem overhead de serialização, agendamento de Tasks ou shuffle de rede
start = time.perf_counter()
resultado_pandas = pdf.groupby("ano")["valor"].sum().sort_values(ascending=False)
pandas_segundos = time.perf_counter() - start

# Spark: mesmo cálculo, mas distribuído (overhead de scheduler + serialização + shuffle)
start = time.perf_counter()
resultado_spark = sdf_vendas.groupBy("ano").agg({"valor": "sum"}).collect()
spark_segundos = time.perf_counter() - start

print(f"Pandas groupBy: {pandas_segundos:.4f}s")
print(f"Spark groupBy:  {spark_segundos:.4f}s")
print()
print("Nesta escala, o menor overhead geralmente vence — em volumes bem maiores,")
print("distribuídos entre múltiplos nós, essa mesma comparação muda de lado.")

📌 **Conclusão da comparação**:

O Pandas venceu nesta escala (~500k linhas) porque:
1. **Sem serialização**: dados já estão em formato numpy na memória
2. **Sem agendamento**: não há Tasks para distribuir entre workers
3. **Sem shuffle**: tudo acontece em um único processo

💡 **Quando o Spark ganha?** Quando o dataset não cabe na memória de uma máquina 
(centenas de milhões a bilhões de linhas) ou quando o processamento exige 
múltiplos estágios de shuffle complexo.

⚠️ **Atenção**: `toPandas()` é perigoso em produção com datasets grandes — pode 
causar `OutOfMemoryError` no driver. Use apenas para amostras pequenas.

In [ ]:
# Encerra a SparkSession e libera todos os recursos (memória, threads, conexões JVM)
# Importante para não deixar processos órfãos no sistema
spark.stop()